# Memory Optimization for iCoExpNet Experiments

## Motivation

When loading multiple GraphToolExperiment objects using `load_hsbm_exps`, memory usage becomes a significant bottleneck. Large experiment sets can consume several gigabytes of RAM, making analysis impractical on resource-constrained systems.

### Key Issues:
1. **Large TPM DataFrames**: Gene expression data (5000 genes × 400+ samples) = 20-100+ MB per experiment
2. **Duplicate Graph Objects**: Both igraph and graph-tool representations stored simultaneously
3. **Inefficient Data Types**: float64 instead of float32, object strings instead of categories
4. **Redundant State Data**: Full partition probabilities vs essential community assignments
5. **Memory Accumulation**: All experiments loaded simultaneously instead of on-demand

### Optimization Goals:
- **Reduce memory footprint by 50-80%**
- **Track optimization improvements systematically**
- **Maintain full functionality for analysis**
- **Create reusable optimization pipeline**

---

## Setup & Imports

In [1]:
# Standard imports

from icoexpnet.analysis.utilities.helpers import save_fig, survival_plot
from icoexpnet.analysis.utilities import sankey_consensus_plot as sky
from icoexpnet.analysis.utilities import clustering as cs
from icoexpnet.analysis import GraphHelper as gh
from icoexpnet.analysis.GraphToolExp import GraphToolExperiment as GtExp
from icoexpnet.analysis.ExperimentSet import ExperimentSet
import os
import sys
import gc
import psutil
import datetime
import json
import pickle
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from collections import defaultdict

# iCoExpNet imports
import sys
sys.path.append('../..')


print("Imports completed successfully")
print(f"Python version: {sys.version}")
print(f"Available RAM: {psutil.virtual_memory().available / (1024**3):.1f} GB")

Imports completed successfully
Python version: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]
Available RAM: 17.9 GB


In [2]:
results_path = "../../../results/"
data_base = "../../../data/"
base_path = "../../../"
test_exps_path = "results/test/"
test_cltrs_path = "results/testCtrl/"

figures_path = f'{results_path}/memory_optimisation/'

mut_df = pd.read_csv(f"{data_base}/test_mutation_data.tsv",
                     sep="\t", index_col="gene")

log_file_path = f'{results_path}/memory_optimisation/memory_optimization_log_v1.1.tsv'


# tf list
tf_path = f"{data_base}/TF_names_v_1.01.txt"
if os.path.exists(tf_path):
    tf_list = np.genfromtxt(fname=tf_path, delimiter="\t",
                            skip_header=1, dtype="str")

## Memory Tracking System

Comprehensive system to track memory usage before/after each optimization step.

In [3]:
class MemoryOptimizationTracker:
    """Track memory usage improvements across optimization steps"""

    def __init__(self, log_file="memory_optimization_log.tsv", experiments_dict=None):
        self.log_file = log_file
        self.experiments = experiments_dict or {}
        self.baseline_recorded = False
        self.current_step = 0

        # Initialize log file with headers if it doesn't exist
        if not os.path.exists(log_file):
            self._initialize_log_file()

    def _initialize_log_file(self):
        """Create TSV file with headers"""
        headers = [
            "timestamp", "step_number", "optimization_type", "description",
            "experiment_set_type", "experiment_name", "experiment_size_mb",
            "set_total_experiments", "set_avg_exp_size_mb", "set_total_size_mb",
            "system_ram_usage_mb", "improvement_vs_baseline_mb",
            "improvement_vs_baseline_percent", "top_memory_consumers",
            "notes", "optimization_details"
        ]
        with open(self.log_file, 'w') as f:
            f.write('\t'.join(headers) + '\n')
        print(f"Created memory optimization log: {self.log_file}")

    def _get_object_size(self, obj, seen=None):
        """Calculate deep object size"""
        size = sys.getsizeof(obj)
        if seen is None:
            seen = set()

        obj_id = id(obj)
        if obj_id in seen:
            return 0

        seen.add(obj_id)

        if isinstance(obj, dict):
            size += sum([self._get_object_size(v, seen) for v in obj.values()])
            size += sum([self._get_object_size(k, seen) for k in obj.keys()])
        elif hasattr(obj, '__dict__'):
            size += self._get_object_size(obj.__dict__, seen)
        elif hasattr(obj, '__iter__') and not isinstance(obj, (str, bytes, bytearray)):
            size += sum([self._get_object_size(i, seen) for i in obj])

        return size

    def _analyze_experiment_memory(self, exp):
        """Get detailed memory breakdown for a single experiment"""
        attrs = []
        for attr_name in dir(exp):
            if not attr_name.startswith('_') and hasattr(exp, attr_name):
                try:
                    attr_value = getattr(exp, attr_name)
                    if not callable(attr_value):
                        size_bytes = self._get_object_size(attr_value)
                        size_mb = size_bytes / (1024 * 1024)
                        attrs.append(
                            (attr_name, size_bytes, size_mb, type(attr_value).__name__))
                except:
                    continue

        # Sort by size
        attrs.sort(key=lambda x: x[1], reverse=True)
        return attrs

    def _get_system_memory_usage(self):
        """Get current system RAM usage"""
        process = psutil.Process(os.getpid())
        return process.memory_info().rss / (1024 * 1024)  # MB

    def record_memory_state(self, optimization_type="baseline", description="Initial state",
                            notes="", optimization_details=None):
        """Record current memory state of all experiments - ONE ROW PER EXPERIMENT"""

        if not self.experiments:
            print("Warning: No experiments provided to track")
            return

        print(f"Recording memory state: {optimization_type}")

        # Force garbage collection
        gc.collect()

        # Calculate set-level statistics first
        total_size_mb = 0
        individual_exp_sizes = {}
        all_top_consumers = []

        for exp_name, exp in self.experiments.items():
            exp_attrs = self._analyze_experiment_memory(exp)
            exp_size_mb = sum(attr[2] for attr in exp_attrs)
            individual_exp_sizes[exp_name] = exp_size_mb
            total_size_mb += exp_size_mb

            # Get top 3 consumers for this experiment
            top_3 = exp_attrs[:3]
            all_top_consumers.extend(
                [f"{exp_name}:{attr[0]}({attr[2]:.1f}MB)" for attr in top_3])

        # Calculate set averages
        num_experiments = len(self.experiments)
        avg_exp_size_mb = total_size_mb / num_experiments if num_experiments > 0 else 0

        # Get system memory usage
        system_ram_mb = self._get_system_memory_usage()

        # Calculate improvements vs baseline
        improvement_mb = 0
        improvement_percent = 0
        if hasattr(self, 'baseline_total_mb'):
            improvement_mb = self.baseline_total_mb - total_size_mb
            improvement_percent = (
                improvement_mb / self.baseline_total_mb) * 100 if self.baseline_total_mb > 0 else 0
        else:
            # This is the baseline
            self.baseline_total_mb = total_size_mb

        # Determine experiment set type
        experiment_set_type = "control" if "ctrl" in optimization_type else "main"

        # Prepare optimization details
        opt_details_str = json.dumps(
            optimization_details) if optimization_details else ""

        # Prepare top consumers string (limit to top 10)
        top_consumers_str = "; ".join(all_top_consumers[:10])

        # Record timestamp
        timestamp = datetime.datetime.now().isoformat()

        # Write ONE ROW PER EXPERIMENT
        for exp_name, exp_size_mb in individual_exp_sizes.items():
            # Calculate individual experiment improvement
            individual_improvement_mb = 0
            individual_improvement_percent = 0
            if hasattr(self, 'baseline_individual') and exp_name in self.baseline_individual:
                individual_improvement_mb = self.baseline_individual[exp_name] - exp_size_mb
                individual_improvement_percent = (
                    individual_improvement_mb / self.baseline_individual[exp_name]) * 100 if self.baseline_individual[exp_name] > 0 else 0
            else:
                # Store baseline for individual experiments
                if not hasattr(self, 'baseline_individual'):
                    self.baseline_individual = {}
                self.baseline_individual[exp_name] = exp_size_mb

            row = [
                timestamp, self.current_step, optimization_type, description,
                experiment_set_type, exp_name, f"{exp_size_mb:.2f}",
                num_experiments, f"{avg_exp_size_mb:.2f}", f"{total_size_mb:.2f}",
                f"{system_ram_mb:.2f}", f"{individual_improvement_mb:.2f}",
                f"{individual_improvement_percent:.2f}", top_consumers_str,
                notes, opt_details_str
            ]

            with open(self.log_file, 'a') as f:
                f.write('\t'.join(str(x) for x in row) + '\n')

        # Print summary
        print(f"Step {self.current_step}: {optimization_type}")
        print(f"  Set total memory: {total_size_mb:.2f} MB")
        print(f"  Set average per experiment: {avg_exp_size_mb:.2f} MB")
        print(
            f" Individual experiment sizes: {min(individual_exp_sizes.values()):.1f} - {max(individual_exp_sizes.values()):.1f} MB")
        print(f"  System RAM: {system_ram_mb:.2f} MB")
        if hasattr(self, 'baseline_total_mb'):
            print(
                f"  Set improvement vs baseline: {improvement_mb:.2f} MB ({improvement_percent:.1f}%)")
        print(
            f"  Logged {len(individual_exp_sizes)} individual experiments to: {self.log_file}")

        self.current_step += 1
        return {
            'total_size_mb': total_size_mb,
            'avg_exp_size_mb': avg_exp_size_mb,
            'individual_sizes': individual_exp_sizes,
            'improvement_mb': improvement_mb,
            'improvement_percent': improvement_percent
        }

    def get_detailed_analysis(self, exp_name):
        """Get detailed memory analysis for a specific experiment"""
        if exp_name not in self.experiments:
            print(f"Experiment {exp_name} not found")
            return None

        exp = self.experiments[exp_name]
        attrs = self._analyze_experiment_memory(exp)

        print(f"\n=== Detailed Memory Analysis: {exp_name} ===")
        print(f"{'Attribute':<20} {'Size (MB)':<12} {'Size (bytes)':<15} {'Type':<20}")
        print("-" * 75)

        for attr_name, size_bytes, size_mb, attr_type in attrs[:15]:
            if size_mb > 0.1:
                print(
                    f"{attr_name:<20} {size_mb:<12.2f} {size_bytes:<15,} {attr_type:<20}")

        return attrs


print("MemoryOptimizationTracker class defined successfully")

MemoryOptimizationTracker class defined successfully


# Load experiments

In [4]:
exp_test = ExperimentSet("test", base_path=base_path, exp_path=test_exps_path,
                         mut_df=mut_df, sel_sets=None, exp_type="iNet")

##### Experiment labels:  dict_keys(['standard_5K_4TF_hsbm', 'standard_5K_3TF_hsbm', 'standard_5K_7TF_hsbm', 'standard_5K_9TF_hsbm', 'standard_5K_5TF_hsbm', 'standard_5K_8TF_hsbm', 'standard_5K_6TF_hsbm'])


In [5]:
# This gets the list of folders of the control experiments
folders = next(os.walk(base_path + test_cltrs_path), (None, None, []))[1]

# Create a dictionary of the control experiments where each key is the index of the control
test_ctrls = {}
for folder in folders:
    hCtrl_path = f"{test_cltrs_path}/{folder}/"
    idx = int(folder.split("tctrl_")[-1])
    test_ctrls[idx] = ExperimentSet(
        "tCtrl", base_path, hCtrl_path, mut_df, sel_sets=None, rel_path="../", exp_type="iNet")
    test_ctrls[idx].export_to_gephi(save=False)

##### Experiment labels:  dict_keys(['standard_5K_3TF_hsbm', 'standard_5K_4TF_hsbm', 'standard_5K_5TF_hsbm', 'standard_5K_6TF_hsbm', 'standard_5K_7TF_hsbm', 'standard_5K_8TF_hsbm', 'standard_5K_9TF_hsbm'])
##### Experiment labels:  dict_keys(['standard_5K_3TF_hsbm', 'standard_5K_5TF_hsbm', 'standard_5K_6TF_hsbm', 'standard_5K_4TF_hsbm', 'standard_5K_7TF_hsbm', 'standard_5K_8TF_hsbm', 'standard_5K_9TF_hsbm'])
##### Experiment labels:  dict_keys(['standard_5K_3TF_hsbm', 'standard_5K_4TF_hsbm', 'standard_5K_5TF_hsbm', 'standard_5K_6TF_hsbm', 'standard_5K_7TF_hsbm', 'standard_5K_8TF_hsbm', 'standard_5K_9TF_hsbm'])
##### Experiment labels:  dict_keys(['standard_5K_3TF_hsbm', 'standard_5K_4TF_hsbm', 'standard_5K_7TF_hsbm', 'standard_5K_5TF_hsbm', 'standard_5K_9TF_hsbm', 'standard_5K_6TF_hsbm', 'standard_5K_8TF_hsbm'])
##### Experiment labels:  dict_keys(['standard_5K_4TF_hsbm', 'standard_5K_3TF_hsbm', 'standard_5K_5TF_hsbm', 'standard_5K_9TF_hsbm', 'standard_5K_6TF_hsbm', 'standard_5K_7TF_hs

# Load the hsbm experiments

In [6]:
# Load the hsbm experiments with the real biological TFs
h_exps, h_entropy = GtExp.load_hsbm_exps(exp_test)
h_entropy["Type"] = "Experiment"

Loading Graph-Tool for standard_5K_4TF_hsbm
Loading Graph-Tool for standard_5K_3TF_hsbm
Loading Graph-Tool for standard_5K_7TF_hsbm
Loading Graph-Tool for standard_5K_9TF_hsbm
Loading Graph-Tool for standard_5K_5TF_hsbm
Loading Graph-Tool for standard_5K_8TF_hsbm
Loading Graph-Tool for standard_5K_6TF_hsbm


In [7]:
# Load the hCtrl experiments with control TFs
ctrl_exps, cmb_df = {}, pd.DataFrame()

# Iterate over the control experiments
for key in range(1, len(test_ctrls) + 1, 1):
    print(f"-->Loading control experiment #{key}")
    exps, entropy = GtExp.load_hsbm_exps(test_ctrls[key])
    entropy["Type"] = "hCtrl{}".format(key)
    cmb_df = pd.concat([cmb_df, entropy], axis=0)
    ctrl_exps[key] = {"entropy": entropy, "exps": exps}

-->Loading control experiment #1
Loading Graph-Tool for standard_5K_4TF_hsbm
Loading Graph-Tool for standard_5K_3TF_hsbm
Loading Graph-Tool for standard_5K_5TF_hsbm
Loading Graph-Tool for standard_5K_9TF_hsbm
Loading Graph-Tool for standard_5K_6TF_hsbm
Loading Graph-Tool for standard_5K_7TF_hsbm
Loading Graph-Tool for standard_5K_8TF_hsbm
-->Loading control experiment #2
Loading Graph-Tool for standard_5K_3TF_hsbm
Loading Graph-Tool for standard_5K_4TF_hsbm
Loading Graph-Tool for standard_5K_5TF_hsbm
Loading Graph-Tool for standard_5K_6TF_hsbm
Loading Graph-Tool for standard_5K_7TF_hsbm
Loading Graph-Tool for standard_5K_8TF_hsbm
Loading Graph-Tool for standard_5K_9TF_hsbm
-->Loading control experiment #3
Loading Graph-Tool for standard_5K_3TF_hsbm
Loading Graph-Tool for standard_5K_5TF_hsbm
Loading Graph-Tool for standard_5K_6TF_hsbm
Loading Graph-Tool for standard_5K_4TF_hsbm
Loading Graph-Tool for standard_5K_7TF_hsbm
Loading Graph-Tool for standard_5K_8TF_hsbm
Loading Graph-Tool fo

## Baseline Memory Measurement

Record initial memory usage before any optimizations.

In [8]:
# Record baseline for main experiments
if len(h_exps) > 0:
    # Initialize the tracker for main experiments
    tracker = MemoryOptimizationTracker(log_file_path, h_exps)

    print("=== Recording Main Experiments Baseline ===")
    baseline_result = tracker.record_memory_state(
        optimization_type="baseline_main",
        description="Initial state before any optimizations - Main experiments",
        notes="Main experiments baseline"
    )

    print(f"\n📊 MAIN EXPERIMENTS BASELINE:")
    print(f"Total memory usage: {baseline_result['total_size_mb']:.2f} MB")
    print(
        f"Average per experiment: {baseline_result['avg_exp_size_mb']:.2f} MB")
    print(f"Number of experiments: {len(h_exps)}")

    # Show detailed analysis for first experiment
    if h_exps:
        first_exp_name = list(h_exps.keys())[0]
        print(f"\n🔍 Detailed analysis for {first_exp_name}:")
        detailed_attrs = tracker.get_detailed_analysis(first_exp_name)

    baseline_recorded = True
    print(f"\n✅ Main experiments baseline recorded!")
else:
    print("❌ No main experiments loaded.")
    baseline_recorded = False
    tracker = None

# Record baseline for control experiments using the SAME tracker and log file
if 'ctrl_exps' in globals() and len(ctrl_exps) > 0 and tracker is not None:
    print("\n" + "="*60)
    print("=== Recording Control Experiments Baseline ===")

    for ctrl_idx, ctrl_set in ctrl_exps.items():
        if 'exps' in ctrl_set and ctrl_set['exps']:
            print(f"\n--- Control Set {ctrl_idx} ---")

            # Update tracker experiments to point to this control set
            original_experiments = tracker.experiments
            tracker.experiments = ctrl_set['exps']

            ctrl_result = tracker.record_memory_state(
                optimization_type=f"baseline_ctrl_{ctrl_idx}",
                description=f"Initial state for control set {ctrl_idx}",
                notes=f"Control experiments set {ctrl_idx} baseline"
            )

            print(f"📊 CONTROL SET {ctrl_idx} BASELINE:")
            print(f"Total memory usage: {ctrl_result['total_size_mb']:.2f} MB")
            print(
                f"Average per experiment: {ctrl_result['avg_exp_size_mb']:.2f} MB")
            print(f"Number of experiments: {len(ctrl_set['exps'])}")

            # Restore original experiments
            tracker.experiments = original_experiments

    print(f"\n✅ All control baselines recorded in same log file!")
else:
    print("\n📝 No control experiments found or main tracker not available")

print(f"\n" + "="*60)
print(f"🎯 SUMMARY:")
print(f"Main experiments: {'✅ Ready' if baseline_recorded else '❌ Not ready'}")
print(f"Control experiments: ✅ Recorded" if 'ctrl_exps' in globals()
      and len(ctrl_exps) > 0 else "📝 None")
print(f"All data logged to: {log_file_path}")
print(f"Ready for optimization: {'✅ Yes' if baseline_recorded else '❌ No'}")

Created memory optimization log: ../../../results//memory_optimisation/memory_optimization_log_v1.1.tsv
=== Recording Main Experiments Baseline ===
Recording memory state: baseline_main
Step 0: baseline_main
  Set total memory: 322.25 MB
  Set average per experiment: 46.04 MB
 Individual experiment sizes: 45.7 - 46.4 MB
  System RAM: 2643.14 MB
  Set improvement vs baseline: 0.00 MB (0.0%)
  Logged 7 individual experiments to: ../../../results//memory_optimisation/memory_optimization_log_v1.1.tsv

📊 MAIN EXPERIMENTS BASELINE:
Total memory usage: 322.25 MB
Average per experiment: 46.04 MB
Number of experiments: 7

🔍 Detailed analysis for standard_5K_4TF_hsbm:

=== Detailed Memory Analysis: standard_5K_4TF_hsbm ===
Attribute            Size (MB)    Size (bytes)    Type                
---------------------------------------------------------------------------
tpm_df               31.63        33,162,029      DataFrame           
edges_df             6.00         6,290,566       DataFrame

## Optimization Step 1: Data Type Optimization

Convert data types to more memory-efficient alternatives:
- float64 → float32 (where precision allows)
- object → category (for repetitive strings)
- integer downcasting

In [9]:
def optimize_dataframe_dtypes(df, name="DataFrame"):
    """Optimize data types for a pandas DataFrame"""
    if df is None or df.empty:
        return df, [], 0, 0, 0

    original_memory = df.memory_usage(deep=True).sum() / 1024 / 1024
    optimizations = []

    for col in df.columns:
        original_dtype = df[col].dtype
        original_size = df[col].memory_usage(deep=True) / 1024 / 1024

        # Optimize numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            # Try to downcast integers
            if pd.api.types.is_integer_dtype(df[col]):
                new_col = pd.to_numeric(df[col], downcast='integer')
                if new_col.dtype != original_dtype:
                    df[col] = new_col
                    new_size = df[col].memory_usage(deep=True) / 1024 / 1024
                    optimizations.append((col, str(original_dtype), str(new_col.dtype),
                                          original_size, new_size))

            # Try to downcast floats (float64 -> float32 where safe)
            elif pd.api.types.is_float_dtype(df[col]):
                if original_dtype == 'float64':
                    # Check if values fit in float32 range
                    col_min, col_max = df[col].min(), df[col].max()
                    if (not pd.isna(col_min) and not pd.isna(col_max) and
                        col_min >= np.finfo(np.float32).min and
                            col_max <= np.finfo(np.float32).max):
                        df[col] = df[col].astype('float32')
                        new_size = df[col].memory_usage(
                            deep=True) / 1024 / 1024
                        optimizations.append((col, str(original_dtype), 'float32',
                                              original_size, new_size))

        # Optimize string/object columns with categorical
        elif df[col].dtype == 'object':
            unique_ratio = df[col].nunique() / len(df[col])
            if unique_ratio < 0.5:  # Less than 50% unique values
                df[col] = df[col].astype('category')
                new_size = df[col].memory_usage(deep=True) / 1024 / 1024
                optimizations.append((col, 'object', 'category',
                                      original_size, new_size))

    optimized_memory = df.memory_usage(deep=True).sum() / 1024 / 1024
    memory_saved = original_memory - optimized_memory

    return df, optimizations, memory_saved, original_memory, optimized_memory


def apply_data_type_optimizations(experiments_dict, tracker, exp_set_name):
    """Apply data type optimizations to all experiments in a set"""

    if not experiments_dict or not tracker:
        print(
            f"❌ Cannot proceed: experiments_dict or tracker not available for {exp_set_name}")
        return None

    optimization_details = {
        'dataframes_optimized': [],
        'total_memory_saved_mb': 0,
        'optimizations_applied': [],
        'errors_encountered': [],
        'experiments_processed': 0
    }

    print(f"=== 🚀 Data Type Optimization: {exp_set_name} ===")

    for exp_name, exp in experiments_dict.items():
        print(f"\n🔧 Optimizing {exp_name}...")
        exp_optimizations = 0
        exp_memory_saved = 0

        # List of DataFrame attributes to optimize
        df_attributes = ['tpm_df', 'edges_df', 'nodes_df', 'meta_df']

        for attr_name in df_attributes:
            if hasattr(exp, attr_name):
                try:
                    df = getattr(exp, attr_name)
                    if df is not None and not df.empty:
                        optimized_df, opts, memory_saved, orig_mem, opt_mem = optimize_dataframe_dtypes(
                            df, f"{exp_name}.{attr_name}")

                        # Update the experiment's attribute
                        setattr(exp, attr_name, optimized_df)

                        if memory_saved > 0:
                            print(
                                f"  ✅ {attr_name}: {orig_mem:.2f} → {opt_mem:.2f} MB (saved {memory_saved:.2f} MB)")
                            optimization_details['dataframes_optimized'].append(
                                f"{exp_name}.{attr_name}")
                            exp_memory_saved += memory_saved
                            exp_optimizations += len(opts)
                            optimization_details['optimizations_applied'].extend(
                                opts)
                        else:
                            print(
                                f"  ➡️  {attr_name}: {orig_mem:.2f} MB (no optimization possible)")

                except Exception as e:
                    error_msg = f"Error optimizing {exp_name}.{attr_name}: {str(e)}"
                    print(f"  ❌ {error_msg}")
                    optimization_details['errors_encountered'].append(
                        error_msg)

        optimization_details['total_memory_saved_mb'] += exp_memory_saved
        optimization_details['experiments_processed'] += 1

        if exp_optimizations > 0:
            print(
                f"  📊 {exp_name} total saved: {exp_memory_saved:.2f} MB ({exp_optimizations} optimizations)")
        else:
            print(f"  📊 {exp_name}: No optimizations applied")

    print(f"\n=== ✅ Data Type Optimization Complete: {exp_set_name} ===")
    print(
        f"📊 Total memory saved: {optimization_details['total_memory_saved_mb']:.2f} MB")
    print(
        f"📈 DataFrames optimized: {len(optimization_details['dataframes_optimized'])}")
    print(
        f"🔧 Experiments processed: {optimization_details['experiments_processed']}")
    print(
        f"⚠️  Errors encountered: {len(optimization_details['errors_encountered'])}")

    # Record the optimization in tracker (this will log individual experiments)
    result = tracker.record_memory_state(
        optimization_type=f"data_type_optimization",
        description=f"Optimized data types (float64→float32, object→category) - {exp_set_name}",
        notes=f"Processed {optimization_details['experiments_processed']} experiments, optimized {len(optimization_details['dataframes_optimized'])} DataFrames",
        optimization_details=optimization_details
    )

    return optimization_details, result


# Apply data type optimizations to main experiments
if baseline_recorded and tracker is not None:
    print("🚀 Starting Data Type Optimization...")
    print("="*70)

    # Optimize main experiments
    print("\n1️⃣ MAIN EXPERIMENTS")
    dtype_results_main, main_result = apply_data_type_optimizations(
        h_exps, tracker, "Main Experiments")

    # Optimize control experiments
    dtype_results_ctrl = {}
    ctrl_results = {}

    if 'ctrl_exps' in globals() and len(ctrl_exps) > 0:
        for ctrl_idx, ctrl_set in ctrl_exps.items():
            if 'exps' in ctrl_set and ctrl_set['exps']:
                print(f"\n{ctrl_idx+1}️⃣ CONTROL SET {ctrl_idx}")

                # Temporarily update tracker experiments to this control set
                original_experiments = tracker.experiments
                tracker.experiments = ctrl_set['exps']

                dtype_results_ctrl[ctrl_idx], ctrl_results[ctrl_idx] = apply_data_type_optimizations(
                    ctrl_set['exps'], tracker, f"Control Set {ctrl_idx}")

                # Restore original experiments
                tracker.experiments = original_experiments

    # Summary
    print(f"\n" + "="*70)
    print(f"🎉 DATA TYPE OPTIMIZATION SUMMARY")
    print(f"="*70)
    print(
        f"✅ Main experiments: {dtype_results_main['total_memory_saved_mb']:.2f} MB saved")

    total_ctrl_saved = sum(results['total_memory_saved_mb']
                           for results in dtype_results_ctrl.values())
    if dtype_results_ctrl:
        print(
            f"✅ Control experiments: {total_ctrl_saved:.2f} MB saved ({len(dtype_results_ctrl)} sets)")

    total_saved = dtype_results_main['total_memory_saved_mb'] + \
        total_ctrl_saved
    print(f"🎯 TOTAL MEMORY SAVED: {total_saved:.2f} MB")
    print(f"📝 All individual experiment data logged to: {log_file_path}")

    data_type_completed = True

else:
    print("❌ Cannot proceed with data type optimization.")
    print("Please ensure baseline measurements were recorded.")
    data_type_completed = False

🚀 Starting Data Type Optimization...

1️⃣ MAIN EXPERIMENTS
=== 🚀 Data Type Optimization: Main Experiments ===

🔧 Optimizing standard_5K_4TF_hsbm...
  ➡️  tpm_df: 2.19 MB (no optimization possible)
  ✅ edges_df: 2.86 → 1.48 MB (saved 1.37 MB)
  ✅ nodes_df: 1.02 → 0.95 MB (saved 0.07 MB)
  ➡️  meta_df: 0.32 MB (no optimization possible)
  📊 standard_5K_4TF_hsbm total saved: 1.44 MB (8 optimizations)

🔧 Optimizing standard_5K_3TF_hsbm...
  ➡️  tpm_df: 2.19 MB (no optimization possible)
  ✅ edges_df: 2.81 → 1.48 MB (saved 1.34 MB)
  ✅ nodes_df: 1.02 → 0.95 MB (saved 0.07 MB)
  ➡️  meta_df: 0.32 MB (no optimization possible)
  📊 standard_5K_3TF_hsbm total saved: 1.40 MB (8 optimizations)

🔧 Optimizing standard_5K_7TF_hsbm...
  ➡️  tpm_df: 2.19 MB (no optimization possible)
  ✅ edges_df: 3.01 → 1.50 MB (saved 1.52 MB)
  ✅ nodes_df: 1.02 → 0.95 MB (saved 0.07 MB)
  ➡️  meta_df: 0.32 MB (no optimization possible)
  📊 standard_5K_7TF_hsbm total saved: 1.58 MB (8 optimizations)

🔧 Optimizing sta

## Optimization Step 2: Selective Data Retention

Remove redundant and unnecessary data:
- Remove duplicate graph representations
- Keep essential community data only
- Clear intermediate computation results

## Results Analysis & Visualization

Analyze the memory optimization results and create visualizations.

## Summary & Recommendations

Summary of optimization results and recommendations for future use.

## Memory Optimization Analysis Summary

### Key Findings from Bioinformatics Network Engineer Analysis

Based on comprehensive analysis of the iCoExpNet codebase, the following memory optimization opportunities were identified:

### Current State (Baseline Analysis)
- **Main Experiments Memory Usage**: 322.25 MB total (46.04 MB per experiment average)
- **Primary Memory Consumers**:
  1. `tpm_df` (TPM DataFrames): 69% of memory per experiment (~31.6 MB each)
  2. `edges_df`: Network edge data (~6.0 MB each)  
  3. `nodes_df`: Network node data (~4.4 MB each)
  4. `mut_df`: Mutation data (~2.1 MB each)
  5. Graph objects and metadata (~1-2 MB each)

### Priority 1 Optimizations: Quick Wins (Target: 20-30% reduction)

#### 1. Data Type Optimizations (IMPLEMENTED)
- **Status**: ✅ Complete - Applied to main and control experiments
- **Results**: Achieved initial memory savings through:
  - Converting `float64` → `float32` where precision allows
  - Converting `object` → `category` for repetitive strings  
  - Integer downcasting
- **Next Steps**: Extend to correlation matrices and graph objects

#### 2. Optimize Data Loading in main.py (NEXT TO IMPLEMENT)
- **Target**: `src/icoexpnet/core/main.py` - Core pipeline data loading
- **Opportunity**: Apply memory optimization immediately after loading TPM/mutation data
- **Expected Impact**: 15-25% reduction in memory usage
- **Implementation**: 
  - Apply `memory_optimization.py` utilities right after data loading
  - Optimize correlation matrix computation memory usage
  - Reduce unnecessary deep copies in pipeline

#### 3. Reduce Unnecessary Deep Copies
- **Target**: Throughout pipeline, especially in experiment processing
- **Opportunity**: Many operations create duplicate DataFrames unnecessarily
- **Expected Impact**: 10-15% reduction
- **Implementation**: Use views and references where data isn't modified

### Priority 2 Optimizations: Medium-term (Target: 40-60% reduction)

#### 4. Implement Lazy Loading for Experiments
- **Challenge**: Need to load all data first, then extract/analyze important bits
- **Solution**: Load data on-demand with intelligent caching
- **Expected Impact**: 30-50% reduction for multi-experiment analyses

#### 5. Compressed Caching with Parquet
- **Target**: Cache processed correlation matrices in compressed format
- **Expected Impact**: 20-30% reduction in cached data size
- **Implementation**: Replace pickle with Parquet for intermediate results

#### 6. Smart Graph Object Management
- **Target**: Avoid storing both igraph and graph-tool representations
- **Expected Impact**: 10-20% reduction
- **Implementation**: Generate graph objects on-demand based on analysis needs

### Priority 3 Optimizations: Long-term (Target: 60-80% reduction)

#### 7. Streaming Data Processing
- **For**: Very large datasets (>10K genes, >1K samples)
- **Implementation**: Process data in chunks for memory-constrained systems

#### 8. Memory Pool Management
- **Implementation**: Centralized memory management with cleanup routines

### Memory Optimization Framework

The existing `memory_optimization.py` utility provides excellent foundation:
- ✅ Data type optimization functions
- ✅ Memory monitoring capabilities  
- ✅ Integration with core classes
- 🔄 Ready for extension to core pipeline

### Tracking & Measurement System

Comprehensive tracking system implemented:
- **Log File**: `../../../results/memory_optimisation/memory_optimization_log_v1.1.tsv`
- **Baseline Recorded**: 322.25 MB for main experiments
- **Per-Experiment Tracking**: Individual memory usage for each experiment
- **Optimization History**: Step-by-step improvement tracking

### Recommendations for Implementation

1. **Start with main.py optimization** - Highest impact, core pipeline
2. **Focus on TPM DataFrame handling** - 69% of memory usage
3. **Implement correlation matrix memory management**
4. **Extend existing memory_optimization.py utilities**
5. **Maintain scientific integrity** - All optimizations preserve analysis accuracy

### Files Requiring Modification

**Primary Targets**:
- `src/icoexpnet/core/main.py` - Core pipeline optimization
- `src/icoexpnet/analysis/ExperimentSet.py` - Multi-experiment management  
- `src/icoexpnet/analysis/utilities/memory_optimization.py` - Enhanced utilities

**Secondary Targets**:
- `src/icoexpnet/analysis/GraphToolExp.py` - Graph object management
- `src/icoexpnet/analysis/NetworkOutput.py` - Output formatting optimization

The analysis provides a clear roadmap for achieving significant memory reductions while maintaining the scientific rigor of the gene co-expression network analysis pipeline.